In [1]:
import pandas as pd
import re
import numpy as np
import unicodedata
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

RESULT_DIR = PROJECT_ROOT / "src" / "5th_Year_Salary_Analysis" / "Result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1. Build the salary and salary-cap reference table

In this step, we combine the salary-cap value for each season with the player salary table.

The goal is to create one clean salary reference table that includes only the columns needed later in the forecasting workflow. We keep the core identifiers and salary information, and remove extra columns that are not needed for modeling.

The final table should keep only records from seasons **1999 to 2025** and should include the key fields needed for later matching, such as:

- player name
- player ID
- team ID
- season
- salary
- salary cap

In [2]:
# ---------------------------
# 1. read the two files
# ---------------------------
salary_df = pd.read_csv(DATA_DIR / "SalaryData" / "salaries_players_1990_2025_combined.csv")
cap_df = pd.read_csv(PROJECT_ROOT / "Output" / "Player Archetype Analysis" / "SalaryBlock" /  "salary_cap_cleaned.csv")

# ---------------------------
# 2. inspect / standardize season start year
# ---------------------------
# salary table: build season_start_year from season column if needed
# example season like "1999-00"
if "season_start_year" not in salary_df.columns:
    salary_df["season_start_year"] = (
        salary_df["season"]
        .astype(str)
        .str[:4]
        .str.extract(r"(\d{4})")[0]
        .astype(float)
    )

# cap table already should have season_start_year
cap_df["season_start_year"] = pd.to_numeric(cap_df["season_start_year"], errors="coerce")

# ---------------------------
# 3. keep only 1999 to 2025 seasons
# ---------------------------
salary_df = salary_df[
    salary_df["season_start_year"].between(1999, 2025, inclusive="both")
].copy()

cap_df = cap_df[
    cap_df["season_start_year"].between(1999, 2025, inclusive="both")
].copy()

# ---------------------------
# 4. merge salary cap into salary table
# ---------------------------
merged_df = salary_df.merge(
    cap_df[["season_start_year", "salary_cap"]],
    on="season_start_year",
    how="left"
)

# ---------------------------
# 5. keep only needed columns
# ---------------------------
final_df = merged_df[
    ["player_name", "player_id", "season", "team_id", "salary", "salary_cap"]
].copy()

# ---------------------------
# 6. optional: sort nicely
# ---------------------------
final_df = final_df.sort_values(
    by=["season", "player_name", "team_id"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------
# 7. quick checks
# ---------------------------
print("Final rows:", len(final_df))
print("Missing salary_cap rows:", final_df["salary_cap"].isna().sum())
print(final_df.head())

Final rows: 14716
Missing salary_cap rows: 0
      player_name  player_id  season  team_id   salary  salary_cap
0      A.C. Green      134.0    1999       13  1700000  34000000.0
1   A.J. Bramlett        NaN    1999        5   118974  34000000.0
2     Aaron McKie     2631.0    1999       20  1624000  34000000.0
3  Aaron Williams     2222.0    1999       27  1100000  34000000.0
4      Adam Keefe      835.0    1999       26  3000000  34000000.0


# Step 2. Clean and Standardize Player Statistics Files

Next, we load the historical player statistics file and prepare to match it against salary data.

This step focuses on basic preprocessing, including:

- Correcting season labels

- Creating usable year variables

- Calculating performance ratings

- Retaining only season records for players who meet both the required number of games played and playing time

- Removing missing rows to avoid future issues

The output of this step is a cleaned season-level basketball data table that will be matched against salary records.

In [3]:
# read file
final_lstm_predictions = pd.read_csv(DATA_DIR / "final_lstm_predictions_all_players.csv")
df = pd.read_csv(DATA_DIR / "all_player_stats_1999-2025.csv")

# mapping for bad Excel-style season values
season_fix_map = {
    "Jan-00": "2000-01",
    "Feb-01": "2001-02",
    "Mar-02": "2002-03",
    "Apr-03": "2003-04",
    "May-04": "2004-05",
    "Jun-05": "2005-06",
    "Jul-06": "2006-07",
    "Aug-07": "2007-08",
    "Sep-08": "2008-09",
    "Oct-09": "2009-10",
    "Nov-10": "2010-11",
    "Dec-11": "2011-12"
}

# clean season column
df["season"] = df["season"].replace(season_fix_map)

# create year column from the first 4 digits of season
df["year"] = df["season"].str.extract(r"^(\d{4})-").astype(int)

# optional: move year next to season
cols = df.columns.tolist()
season_idx = cols.index("season")
cols.insert(season_idx + 1, cols.pop(cols.index("year")))
df = df[cols]

# check result
print(df[["season", "year"]].drop_duplicates().sort_values("year"))
print(df.head())

        season  year
0      1999-00  1999
439    2000-01  2000
880    2001-02  2001
1320   2002-03  2002
1748   2003-04  2003
2190   2004-05  2004
2654   2005-06  2005
3112   2006-07  2006
3570   2007-08  2007
4021   2008-09  2008
4466   2009-10  2009
4908   2010-11  2010
5360   2011-12  2011
5838   2012-13  2012
6307   2013-14  2013
6789   2014-15  2014
7281   2015-16  2015
7757   2016-17  2016
8243   2017-18  2017
8783   2018-19  2018
9313   2019-20  2019
9842   2020-21  2020
10382  2021-22  2021
10987  2022-23  2022
11526  2023-24  2023
12098  2024-25  2024
12667  2025-26  2025
   PLAYER_ID     PLAYER_NAME NICKNAME     TEAM_ID TEAM_ABBREVIATION  AGE  GP  \
0        920      A.C. Green     A.C.  1610612747               LAL   36  82   
1       1920   A.J. Bramlett     A.J.  1610612739               CLE   23   8   
2        243     Aaron McKie    Aaron  1610612755               PHI   27  82   
3       1425  Aaron Williams    Aaron  1610612764               WAS   28  81   
4        228

In [4]:
# create Game Score / performance score
df["performance_score"] = (
    df["PTS"]
    + 0.4 * df["FGM"]
    - 0.7 * df["FGA"]
    - 0.4 * (df["FTA"] - df["FTM"])
    + 0.7 * df["OREB"]
    + 0.3 * df["DREB"]
    + df["STL"]
    + 0.7 * df["AST"]
    + 0.7 * df["BLK"]
    - 0.4 * df["PF"]
    - df["TOV"]
)

# check result
print(df[["PLAYER_NAME", "season", "year", "performance_score"]].head())

      PLAYER_NAME   season  year  performance_score
0      A.C. Green  1999-00  1999              427.3
1   A.J. Bramlett  1999-00  1999               -0.9
2     Aaron McKie  1999-00  1999              516.1
3  Aaron Williams  1999-00  1999              531.7
4      Adam Keefe  1999-00  1999               89.1


In [5]:
# -----------------------------
# Keep only seasons where player actually played enough
df_clean = df[(df["GP"] > 20) & (df["MIN"] > 10)].copy()

# -----------------------------
# 3. Check missing values
# -----------------------------
print("Missing values by column:")
print(df_clean.isna().sum())

# -----------------------------
# 4. Remove only rows with missing values
# -----------------------------
# This removes only the player-year row that has missing data
df_clean = df_clean.dropna().copy()

print("\nShape after dropping missing rows:")
print(df_clean.shape)

# -----------------------------
# 5. Count number of seasons per player
# -----------------------------
# assuming player name column is PLAYER_NAME
player_season_counts = (
    df_clean.groupby("PLAYER_NAME")
    .size()
    .reset_index(name="n_seasons")
)

# merge season count back to dataframe
df_clean = df_clean.merge(player_season_counts, on="PLAYER_NAME", how="left")

Missing values by column:
PLAYER_ID            0
PLAYER_NAME          0
NICKNAME             0
TEAM_ID              0
TEAM_ABBREVIATION    0
AGE                  0
GP                   0
W                    0
L                    0
W_PCT                0
MIN                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
FG3A                 0
FG3_PCT              0
FTM                  0
FTA                  0
FT_PCT               0
OREB                 0
DREB                 0
REB                  0
AST                  0
TOV                  0
STL                  0
BLK                  0
BLKA                 0
PF                   0
PFD                  0
PTS                  0
PLUS_MINUS           0
NBA_FANTASY_PTS      0
DD2                  0
TD3                  0
WNBA_FANTASY_PTS     0
TEAM_COUNT           0
season               0
year                 0
performance_score    0
dtype: int64

Shape after dropping missing rows:
(1089

# Step 3: Standardize Player Names Before Merging with Salary Data

Before merging the basketball statistics table with the salary table, we first need to clean the player names from both sides.

This is necessary because player names are not always spelled exactly the same in different files. Differences in punctuation, accents, spaces, or encoding can all cause matching failures.

In this step, we:

- Clean player names

- Create a normalized merge key

- Manually handle known special cases

This provides a more reliable name-based matching process for the next merging step.

In [6]:
# --------------------------------------------------
# 1. basic name cleaning
# --------------------------------------------------
salary_match_df = final_df.drop(
    columns=["player_id", "team_id"],
    errors="ignore"
).copy()

def fix_player_name(x):
    if pd.isna(x):
        return x
    
    s = str(x).strip()

    # common bad punctuation / encoding leftovers
    replacements = {
        "’": "'",
        "`": "'",
        "´": "'",
        "“": '"',
        "”": '"',
        "?": "",
    }
    for bad, good in replacements.items():
        s = s.replace(bad, good)

    # remove accents safely
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")

    # clean repeated spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

# --------------------------------------------------
# 2. normalize name into merge key
# --------------------------------------------------
def normalize_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().strip()
    x = re.sub(r"\.", "", x)
    x = re.sub(r"'", "", x)
    x = re.sub(r"[^a-z0-9\s-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# --------------------------------------------------
# 3. apply base cleaning
# --------------------------------------------------
df_clean["PLAYER_NAME_fixed"] = df_clean["PLAYER_NAME"].apply(fix_player_name)
salary_match_df["player_name_fixed"] = salary_match_df["player_name"].apply(fix_player_name)

df_clean["name_key"] = df_clean["PLAYER_NAME_fixed"].apply(normalize_name)
salary_match_df["name_key"] = salary_match_df["player_name_fixed"].apply(normalize_name)

# --------------------------------------------------
# 4. manual corrections for known bad names
# --------------------------------------------------

manual_name_map = {
    # stars / common encoding issues
    "nikola vuevi": "nikola vucevic",
    "luka doni": "luka doncic",
    "nikola joki": "nikola jokic",
    "jusuf nurki": "jusuf nurkic",
    "bogdan bogdanovi": "bogdan bogdanovic",
    "bojan bogdanovi": "bojan bogdanovic",
    "jonas valaniunas": "jonas valanciunas",
    "dario ari": "dario saric",
    "goran dragi": "goran dragic",
    "oran dragi": "zoran dragic",
    "milo teodosi": "milos teodosic",
    "anzejs paseiks": "anzejs pasecniks",
    "clar weatherspoon": "clarence weatherspoon",
    "dennis schrder": "dennis schroder",
    "egor dmin": "egor demin",
    "hugo gonzlez": "hugo gonzalez",
    "hedo turkoglu": "hedo turkoglu",
    "charles r jones": "charles r jones",
    "charlie brown jr": "charlie brown jr",
    "da ron holmes ii": "daron holmes ii",

    # optional suffix consistency
    "andre jackson jr": "andre jackson jr",
    "craig porter jr": "craig porter jr",
    "david duke jr": "david duke jr",
    "derrick walton jr": "derrick walton jr",
    "greg brown iii": "greg brown iii",
    "harry giles iii": "harry giles iii",
    "frank mason iii": "frank mason iii",
}

df_clean["name_key"] = df_clean["name_key"].replace(manual_name_map)
salary_match_df["name_key"] = salary_match_df["name_key"].replace(manual_name_map)

# --------------------------------------------------
# 5. quick check of changed names
# --------------------------------------------------
changed_df_names = df_clean.loc[
    df_clean["PLAYER_NAME"] != df_clean["PLAYER_NAME_fixed"],
    ["PLAYER_NAME", "PLAYER_NAME_fixed", "name_key"]
].drop_duplicates()

changed_salary_names = salary_match_df.loc[
    salary_match_df["player_name"] != salary_match_df["player_name_fixed"],
    ["player_name", "player_name_fixed", "name_key"]
].drop_duplicates()

print("Changed names in all-stats side:")
print(changed_df_names.head(50))

print("\nChanged names in salary side:")
print(changed_salary_names.head(50))

Changed names in all-stats side:
                    PLAYER_NAME        PLAYER_NAME_fixed  \
4866             Nikola Vu?evi?             Nikola Vuevi   
5180          Jonas Valan?i?nas          Jonas Valaninas   
5468            Dennis Schr?der           Dennis Schrder   
6007               Jusuf Nurki?              Jusuf Nurki   
6434         Kristaps Porzi??is         Kristaps Porziis   
6497               Nikola Joki?              Nikola Joki   
7056          Bogdan Bogdanovi?         Bogdan Bogdanovi   
7694                Luka Don?i?                Luka Doni   
7868           An?ejs Pase??iks            Anejs Paseiks   
8670              Vlatko ?an?ar              Vlatko anar   
9112                 V?t Krej??                  Vt Krej   
9438             Moussa Diabat?            Moussa Diabat   
9892               Nikola Jovi?              Nikola Jovi   
10248            Karlo Matkovi?            Karlo Matkovi   
10401            Tidjane Sala?n            Tidjane Salan   
10574  

# Step 4: Merge the Cleaned Salary and Player Statistics Tables

Now we will merge the salary table with the cleaned player statistics table.

Since the player IDs in the two data sources do not match exactly, we will primarily merge based on the cleaned player names. Additionally, we need to carefully align the season information, as one file uses the NBA season format, while the other uses calendar year format season tags.

The main goals of this step are:

- To merge salary information into the basketball statistics table

- To ensure correct season alignment

- To keep the **player statistics table** as the primary table

- To retain all rows in the statistics table, even if some salary matches are missing

In [7]:
# ----------------------------------------
# 1. start from salary table and drop ids
# ----------------------------------------
salary_merge_df = final_df.drop(
    columns=[c for c in ["player_id", "team_id"] if c in final_df.columns],
    errors="ignore"
).copy()

# ----------------------------------------
# 2. make sure the cleaned name columns exist
#    if not, rebuild from earlier fixed columns
# ----------------------------------------
if "player_name_fixed" not in salary_merge_df.columns:
    salary_merge_df["player_name_fixed"] = salary_merge_df["player_name"].apply(fix_player_name)

if "name_key" not in salary_merge_df.columns:
    salary_merge_df["name_key"] = salary_merge_df["player_name_fixed"].apply(normalize_name)
    salary_merge_df["name_key"] = salary_merge_df["name_key"].replace(manual_name_map)

if "PLAYER_NAME_fixed" not in df_clean.columns:
    df_clean["PLAYER_NAME_fixed"] = df_clean["PLAYER_NAME"].apply(fix_player_name)

if "name_key" not in df_clean.columns:
    df_clean["name_key"] = df_clean["PLAYER_NAME_fixed"].apply(normalize_name)
    df_clean["name_key"] = df_clean["name_key"].replace(manual_name_map)

# ----------------------------------------
# 3. create comparable season year
#    all-stats side: use existing year if present
#    salary side: season like 1999-00 -> 1999
# ----------------------------------------
df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce")

salary_merge_df["year"] = (
    salary_merge_df["season"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
    .astype(float)
)

# ----------------------------------------
# 4. keep only needed salary columns
# ----------------------------------------
salary_merge_df = salary_merge_df[
    ["name_key", "player_name_fixed", "season", "year", "salary", "salary_cap"]
].copy()

salary_merge_df = salary_merge_df.rename(columns={"season": "season_salary"})

# ----------------------------------------
# 5. aggregate salary rows to one player-year
#    because a player may have multiple team rows
# ----------------------------------------
salary_merge_df_agg = (
    salary_merge_df
    .groupby(["name_key", "year"], as_index=False)
    .agg({
        "player_name_fixed": "first",
        "season_salary": "first",
        "salary": "sum",
        "salary_cap": "first"
    })
)

# ----------------------------------------
# 6. merge onto all-stats table
#    left join keeps all rows from df_clean
# ----------------------------------------
df_clean_merged = df_clean.merge(
    salary_merge_df_agg,
    on=["name_key", "year"],
    how="left"
)

# ----------------------------------------
# 7. optional season checks
# ----------------------------------------
df_clean_merged["season_match_exact"] = (
    df_clean_merged["season"].astype(str).str.strip()
    == df_clean_merged["season_salary"].astype(str).str.strip()
)

df_clean_merged["salary_year"] = (
    df_clean_merged["season_salary"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
    .astype(float)
)

df_clean_merged["season_match_year"] = (
    df_clean_merged["year"] == df_clean_merged["salary_year"]
)

# ----------------------------------------
# 8. quick checks
# ----------------------------------------
print("Rows in df_clean:", len(df_clean))
print("Rows after merge:", len(df_clean_merged))
print("Matched salary rows:", df_clean_merged["salary"].notna().sum())
print("Unmatched salary rows:", df_clean_merged["salary"].isna().sum())

print("\nExact season match:")
print(df_clean_merged["season_match_exact"].value_counts(dropna=False))

print("\nYear-based season match:")
print(df_clean_merged["season_match_year"].value_counts(dropna=False))

print("\nSample rows:")
print(
    df_clean_merged[
        ["PLAYER_NAME", "PLAYER_NAME_fixed", "season", "year", "season_salary", "salary", "salary_cap"]
    ].head(15)
)

Rows in df_clean: 10892
Rows after merge: 10892
Matched salary rows: 10300
Unmatched salary rows: 592

Exact season match:
season_match_exact
False    10892
Name: count, dtype: int64

Year-based season match:
season_match_year
True     10300
False      592
Name: count, dtype: int64

Sample rows:
          PLAYER_NAME  PLAYER_NAME_fixed   season  year  season_salary  \
0          A.C. Green         A.C. Green  1999-00  1999         1999.0   
1         Aaron McKie        Aaron McKie  1999-00  1999         1999.0   
2      Aaron Williams     Aaron Williams  1999-00  1999         1999.0   
3          Adam Keefe         Adam Keefe  1999-00  1999         1999.0   
4        Adonal Foyle       Adonal Foyle  1999-00  1999         1999.0   
5      Adrian Griffin     Adrian Griffin  1999-00  1999         1999.0   
6       Al Harrington      Al Harrington  1999-00  1999         1999.0   
7      Alan Henderson     Alan Henderson  1999-00  1999         1999.0   
8       Allan Houston      Allan Hous

# Step 5: Check Season Consistency

After merging, we only retain records containing salary information. We then compare the season columns from both data sources to check if the basketball season and salary season are correctly aligned. This is crucial because even if player names match, the seasons may not.

In [8]:
# ----------------------------------------
# 1. keep only rows with non-missing salary
# ----------------------------------------
df_clean_merged_salary = df_clean_merged[df_clean_merged["salary"].notna()].copy()

print("Original shape:", df_clean_merged.shape)
print("After keeping non-missing salary:", df_clean_merged_salary.shape)

# ----------------------------------------
# 2. compare season from df_clean vs season_salary from salary file
# ----------------------------------------
# exact text match
df_clean_merged_salary["season_match_exact"] = (
    df_clean_merged_salary["season"].astype(str).str.strip()
    == df_clean_merged_salary["season_salary"].astype(str).str.strip()
)

# year-based match
df_clean_merged_salary["salary_year"] = (
    df_clean_merged_salary["season_salary"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
    .astype(float)
)

df_clean_merged_salary["season_match_year"] = (
    df_clean_merged_salary["year"] == df_clean_merged_salary["salary_year"]
)

# ----------------------------------------
# 3. summary
# ----------------------------------------
print("\nExact season string match count:")
print(df_clean_merged_salary["season_match_exact"].value_counts(dropna=False))

print("\nYear-based season match count:")
print(df_clean_merged_salary["season_match_year"].value_counts(dropna=False))

# ----------------------------------------
# 4. show mismatches
# ----------------------------------------
season_mismatch = df_clean_merged_salary[
    df_clean_merged_salary["season_match_year"] == False
].copy()

print("\nNumber of year mismatches:", len(season_mismatch))

print(
    season_mismatch[
        ["PLAYER_NAME", "PLAYER_NAME_fixed", "season", "year", "season_salary", "salary_year", "salary", "salary_cap"]
    ].drop_duplicates().head(20)
)

Original shape: (10892, 50)
After keeping non-missing salary: (10300, 50)

Exact season string match count:
season_match_exact
False    10300
Name: count, dtype: int64

Year-based season match count:
season_match_year
True    10300
Name: count, dtype: int64

Number of year mismatches: 0
Empty DataFrame
Columns: [PLAYER_NAME, PLAYER_NAME_fixed, season, year, season_salary, salary_year, salary, salary_cap]
Index: []


# Step 6: Retain Players with Sufficient Historical Data for Prediction

The salary prediction task uses data from a player's **previous four seasons** as input. Therefore, we only retain players with at least **four valid seasons** in the merged dataset.

In [9]:
# ----------------------------------------
# 1. count number of unique years played for each player
# ----------------------------------------
player_year_counts = (
    df_clean_merged_salary
    .groupby("PLAYER_NAME_fixed")["year"]
    .nunique()
    .reset_index(name="n_years_played")
)

print("Player-year count table:")
print(player_year_counts.head())

# ----------------------------------------
# 2. merge count back to dataframe
# ----------------------------------------
df_clean_merged_salary = df_clean_merged_salary.merge(
    player_year_counts,
    on="PLAYER_NAME_fixed",
    how="left"
)

# ----------------------------------------
# 3. keep only players with at least 4 years
# ----------------------------------------
df_clean_merged_salary_4plus = df_clean_merged_salary[
    df_clean_merged_salary["n_years_played"] >= 4
].copy()

# ----------------------------------------
# 4. quick checks
# ----------------------------------------
print("\nOriginal shape:", df_clean_merged_salary.shape)
print("After keeping players with >= 4 years:", df_clean_merged_salary_4plus.shape)

print("\nUnique players before:", df_clean_merged_salary["PLAYER_NAME_fixed"].nunique())
print("Unique players after:", df_clean_merged_salary_4plus["PLAYER_NAME_fixed"].nunique())

print("\nExample rows:")
print(
    df_clean_merged_salary_4plus[
        ["PLAYER_NAME", "PLAYER_NAME_fixed", "year", "season", "salary", "salary_cap", "n_years_played"]
    ].head(20)
)

Player-year count table:
  PLAYER_NAME_fixed  n_years_played
0        A.C. Green               2
1       A.J. Guyton               2
2       A.J. Lawson               2
3          AJ Green               3
4        AJ Griffin               1

Original shape: (10300, 51)
After keeping players with >= 4 years: (8748, 51)

Unique players before: 1924
Unique players after: 1058

Example rows:
          PLAYER_NAME  PLAYER_NAME_fixed  year   season      salary  \
1         Aaron McKie        Aaron McKie  1999  1999-00   1624000.0   
2      Aaron Williams     Aaron Williams  1999  1999-00   1100000.0   
4        Adonal Foyle       Adonal Foyle  1999  1999-00   1922520.0   
5      Adrian Griffin     Adrian Griffin  1999  1999-00    385000.0   
6       Al Harrington      Al Harrington  1999  1999-00    745800.0   
7      Alan Henderson     Alan Henderson  1999  1999-00   5318437.0   
8       Allan Houston      Allan Houston  1999  1999-00   8000000.0   
9       Allen Iverson      Allen Iverson 

# Step 7: Clean up player data tables

Now, we simplify the merged table by removing columns used only for matching or diagnosis. Including: NICKNAME, name key, season_salary, and salary year, season_match_exact and season_match_year. We also set the cleaned-up player name field as the primary name field used in subsequent steps.

After cleaning the table, we reorder the columns so the most important identifiers appear first.

In [10]:
# ----------------------------------------
# final cleanup for 4+ years table
# ----------------------------------------

# 1. drop unwanted columns
cols_to_drop = [
    "NICKNAME",
    "name_key",
    "season_salary",
    "salary_year",
    "season_match_exact",
    "season_match_year",
    "season",
]

df_clean_merged_salary_4plus = df_clean_merged_salary_4plus.drop(
    columns=cols_to_drop,
    errors="ignore"
)

# 2. move PLAYER_NAME_fixed to second column and year to third column
cols = df_clean_merged_salary_4plus.columns.tolist()

# remove them first if they exist
for c in ["player_name_fixed", "year"]:
    if c in cols:
        cols.remove(c)

# keep first column as it is, then insert PLAYER_NAME_fixed and year
new_cols = [cols[0]]

if "player_name_fixed" in df_clean_merged_salary_4plus.columns:
    new_cols.append("player_name_fixed")

if "year" in df_clean_merged_salary_4plus.columns:
    new_cols.append("year")

# add the rest
new_cols += cols[1:]

df_clean_merged_salary_4plus = df_clean_merged_salary_4plus[new_cols]

# 3. quick check
print(df_clean_merged_salary_4plus.columns.tolist())
print(df_clean_merged_salary_4plus.head())
print(df_clean_merged_salary_4plus.shape)

['PLAYER_ID', 'player_name_fixed', 'year', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'TEAM_COUNT', 'performance_score', 'n_seasons', 'PLAYER_NAME_fixed', 'salary', 'salary_cap', 'n_years_played']
   PLAYER_ID player_name_fixed  year     PLAYER_NAME     TEAM_ID  \
1        243       Aaron McKie  1999     Aaron McKie  1610612755   
2       1425    Aaron Williams  1999  Aaron Williams  1610612764   
4       1502      Adonal Foyle  1999    Adonal Foyle  1610612744   
5       1559    Adrian Griffin  1999  Adrian Griffin  1610612738   
6       1733     Al Harrington  1999   Al Harrington  1610612754   

  TEAM_ABBREVIATION  AGE  GP   W   L  ...  DD2  TD3  WNBA_FANTASY_PTS  \
1               PHI   27  82  49  33  ...    0    0      

In [11]:
# ----------------------------------------
# clean duplicate / unwanted player columns
# ----------------------------------------

# 1. drop unwanted columns
cols_to_drop = ["n_seasons", "PLAYER_NAME"]
df_clean_merged_salary_4plus = df_clean_merged_salary_4plus.drop(
    columns=cols_to_drop,
    errors="ignore"
)

# 2. if both PLAYER_NAME_fixed and player_name_fixed exist, keep only one
if "PLAYER_NAME_fixed" in df_clean_merged_salary_4plus.columns and "player_name_fixed" in df_clean_merged_salary_4plus.columns:
    df_clean_merged_salary_4plus = df_clean_merged_salary_4plus.drop(
        columns=["PLAYER_NAME_fixed"],
        errors="ignore"
    )

# 3. if only lowercase exists, rename it to uppercase version
elif "player_name_fixed" in df_clean_merged_salary_4plus.columns and "PLAYER_NAME_fixed" not in df_clean_merged_salary_4plus.columns:
    df_clean_merged_salary_4plus = df_clean_merged_salary_4plus.rename(
        columns={"player_name_fixed": "PLAYER_NAME_fixed"}
    )

# 5. check result
print(df_clean_merged_salary_4plus.columns.tolist())
print(df_clean_merged_salary_4plus.head())
print(df_clean_merged_salary_4plus.shape)

['PLAYER_ID', 'player_name_fixed', 'year', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'TEAM_COUNT', 'performance_score', 'salary', 'salary_cap', 'n_years_played']
   PLAYER_ID player_name_fixed  year     TEAM_ID TEAM_ABBREVIATION  AGE  GP  \
1        243       Aaron McKie  1999  1610612755               PHI   27  82   
2       1425    Aaron Williams  1999  1610612764               WAS   28  81   
4       1502      Adonal Foyle  1999  1610612744               GSW   25  76   
5       1559    Adrian Griffin  1999  1610612738               BOS   25  72   
6       1733     Al Harrington  1999  1610612754               IND   20  50   

    W   L  W_PCT  ...  PLUS_MINUS  NBA_FANTASY_PTS  DD2  TD3  \
1  49  33  0.598  ...          52           1573.

# Step 8. Construct the Input Table for the First Four Years of Prediction

The final prediction task is to predict the **salary cap percentage for the fifth year**, therefore we need players with at least **five years of valid data**.

In this step, we:

- Only retain players with at least five seasons of data

- Sort each player by year

- Only retain the **first four seasons** of data for each player

In [12]:
# ----------------------------------------
# 1. count unique years per player
# ----------------------------------------
player_year_counts_5plus = (
    df_clean_merged_salary_4plus
    .groupby("player_name_fixed")["year"]
    .nunique()
    .reset_index(name="n_years_played")
)

# keep only players with at least 5 years
players_5plus = player_year_counts_5plus.loc[
    player_year_counts_5plus["n_years_played"] >= 5,
    "player_name_fixed"
]

df_5plus = df_clean_merged_salary_4plus[
    df_clean_merged_salary_4plus["player_name_fixed"].isin(players_5plus)
].copy()

print("Shape after keeping players with >= 5 years:", df_5plus.shape)
print("Unique players with >= 5 years:", df_5plus["player_name_fixed"].nunique())

# ----------------------------------------
# 2. sort by player and year
# ----------------------------------------
df_5plus = df_5plus.sort_values(
    by=["player_name_fixed", "year"]
).copy()

# ----------------------------------------
# 3. keep only first 4 years for each player
# ----------------------------------------
df_first4 = (
    df_5plus
    .groupby("player_name_fixed", group_keys=False)
    .head(4)
    .copy()
)

print("Shape after keeping first 4 years only:", df_first4.shape)
print("Unique players in final table:", df_first4["player_name_fixed"].nunique())

# ----------------------------------------
# 4. quick check
# ----------------------------------------
print(df_first4[["player_name_fixed", "year"]].head(20))

# verify each player has exactly 4 rows
check_counts = (
    df_first4.groupby("player_name_fixed")
    .size()
    .reset_index(name="n_rows_kept")
)

print(check_counts["n_rows_kept"].value_counts())

Shape after keeping players with >= 5 years: (8112, 41)
Unique players with >= 5 years: 899
Shape after keeping first 4 years only: (3596, 41)
Unique players in final table: 899
     player_name_fixed  year
3665          AJ Price  2009
4040          AJ Price  2010
4425          AJ Price  2011
4808          AJ Price  2012
2923      Aaron Brooks  2007
3296      Aaron Brooks  2008
3666      Aaron Brooks  2009
4041      Aaron Brooks  2010
5583      Aaron Gordon  2014
5982      Aaron Gordon  2015
6372      Aaron Gordon  2016
6759      Aaron Gordon  2017
2924        Aaron Gray  2007
3297        Aaron Gray  2008
3667        Aaron Gray  2009
4042        Aaron Gray  2010
7143     Aaron Holiday  2018
7535     Aaron Holiday  2019
7898     Aaron Holiday  2020
8299     Aaron Holiday  2021
n_rows_kept
4    899
Name: count, dtype: int64


# Step 9. Add Player Performance Category Labels to the "First Four Years" Table

Next, we enrich the "First Four Years" prediction table using the final player classification results from the clustering section.

We incorporate the output of `final_lstm_predictions_all_players`, which includes the following variables:

- Predicted Category

- Category Probability

- Maximum Predicted Probability

In [13]:
# ----------------------------------------
# 1. make column names lowercase
# ----------------------------------------
final_lstm_predictions.columns = final_lstm_predictions.columns.str.strip().str.lower()
df_first4.columns = df_first4.columns.str.strip().str.lower()

# ----------------------------------------
# 2. choose columns from goal 1
# ----------------------------------------
goal1_cols = [
    "player_id",
    "predicted_class",
    "prob_bust",
    "prob_neutral",
    "prob_sleeper",
    "max_pred_prob"
]

goal1_cols = [c for c in goal1_cols if c in final_lstm_predictions.columns]
goal1_player = final_lstm_predictions[goal1_cols].copy()

# keep one row per player_id
goal1_player = goal1_player.drop_duplicates(subset=["player_id"]).copy()

print("Rows in goal1_player:", len(goal1_player))
print("Unique player_id in goal1_player:", goal1_player["player_id"].nunique())

# ----------------------------------------
# 3. merge by player_id
# ----------------------------------------
df_first4_with_label = df_first4.merge(
    goal1_player,
    on="player_id",
    how="left"
)

# ----------------------------------------
# 4. quick checks
# ----------------------------------------
print("Rows in df_first4:", len(df_first4))
print("Rows after merge:", len(df_first4_with_label))
print("Matched predicted_class rows:", df_first4_with_label["predicted_class"].notna().sum())
print("Unmatched predicted_class rows:", df_first4_with_label["predicted_class"].isna().sum())

print(
    df_first4_with_label[
        [
            "player_id",
            "player_name_fixed",
            "year",
            "predicted_class",
            "prob_bust",
            "prob_neutral",
            "prob_sleeper"
        ]
    ].head(20)
)

Rows in goal1_player: 1121
Unique player_id in goal1_player: 1121
Rows in df_first4: 3596
Rows after merge: 3596
Matched predicted_class rows: 3596
Unmatched predicted_class rows: 0
    player_id player_name_fixed  year predicted_class  prob_bust  \
0      201985          AJ Price  2009            Bust   0.871164   
1      201985          AJ Price  2010            Bust   0.871164   
2      201985          AJ Price  2011            Bust   0.871164   
3      201985          AJ Price  2012            Bust   0.871164   
4      201166      Aaron Brooks  2007         Neutral   0.277595   
5      201166      Aaron Brooks  2008         Neutral   0.277595   
6      201166      Aaron Brooks  2009         Neutral   0.277595   
7      201166      Aaron Brooks  2010         Neutral   0.277595   
8      203932      Aaron Gordon  2014         Neutral   0.033125   
9      203932      Aaron Gordon  2015         Neutral   0.033125   
10     203932      Aaron Gordon  2016         Neutral   0.033125   
11

In [14]:
# ----------------------------------------
# 5. show all unmatched rows
# ----------------------------------------
unmatched_rows = df_first4_with_label[
    df_first4_with_label["predicted_class"].isna()
].copy()

print("Number of unmatched rows:", len(unmatched_rows))
print("Number of unmatched players:", unmatched_rows["player_id"].nunique())

print(unmatched_rows)

Number of unmatched rows: 0
Number of unmatched players: 0
Empty DataFrame
Columns: [player_id, player_name_fixed, year, team_id, team_abbreviation, age, gp, w, l, w_pct, min, fgm, fga, fg_pct, fg3m, fg3a, fg3_pct, ftm, fta, ft_pct, oreb, dreb, reb, ast, tov, stl, blk, blka, pf, pfd, pts, plus_minus, nba_fantasy_pts, dd2, td3, wnba_fantasy_pts, team_count, performance_score, salary, salary_cap, n_years_played, predicted_class, prob_bust, prob_neutral, prob_sleeper, max_pred_prob]
Index: []

[0 rows x 46 columns]


# Step 10. Only keep rows with successfully matched category labels

After merging the label tables, some rows may still remain unmatched. To maintain the simplicity and consistency of the prediction table, we remove rows without valid predicted categories.


In [15]:
# keep only matched rows
df_first4_with_label_matched = df_first4_with_label[
    df_first4_with_label["predicted_class"].notna()
].copy()

print("Original shape:", df_first4_with_label.shape)
print("After removing unmatched rows:", df_first4_with_label_matched.shape)

print("Unique players after removing unmatched rows:",
      df_first4_with_label_matched["player_id"].nunique())

Original shape: (3596, 46)
After removing unmatched rows: (3596, 46)
Unique players after removing unmatched rows: 899


# Step 11: Adding Prototype Information to the Prediction Table

Next, we use `player_id` to incorporate player prototype information into the working dataset.

The two main fields added here are:

- `macro_archetype`

- `shot_style_subtype`

In [16]:
# ----------------------------------------
# 1. read / prepare block2 file if needed
# ----------------------------------------
block2 = pd.read_csv(PROJECT_ROOT / "Output" / "Player Archetype Analysis" / "SalaryBlock" /  "deliverable3_block2_archetype_comp_market_context.csv")

# make column names lowercase
block2.columns = block2.columns.str.strip().str.lower()
df_first4_with_label_matched.columns = df_first4_with_label_matched.columns.str.strip().str.lower()

# ----------------------------------------
# 2. keep only needed columns from block2
# ----------------------------------------
block2_keep = [
    "player_id",
    "macro_archetype",
    "shot_style_subtype"
]

block2_keep = [c for c in block2_keep if c in block2.columns]
block2_small = block2[block2_keep].copy()

# keep one row per player_id
block2_small = block2_small.drop_duplicates(subset=["player_id"]).copy()

print("Rows in block2_small:", len(block2_small))
print("Unique players in block2_small:", block2_small["player_id"].nunique())

# ----------------------------------------
# 3. merge by player_id
# ----------------------------------------
df_with_archetype = df_first4_with_label_matched.merge(
    block2_small,
    on="player_id",
    how="left"
)

# ----------------------------------------
# 4. matched row counts
# ----------------------------------------
matched_rows = df_with_archetype["macro_archetype"].notna().sum()
unmatched_rows = df_with_archetype["macro_archetype"].isna().sum()

print("\nRow-level match result:")
print("Total rows:", len(df_with_archetype))
print("Matched rows:", matched_rows)
print("Unmatched rows:", unmatched_rows)

# ----------------------------------------
# 5. matched player counts
# ----------------------------------------
player_match_check = (
    df_with_archetype.groupby("player_id")["macro_archetype"]
    .apply(lambda x: x.notna().any())
    .reset_index(name="matched_flag")
)

matched_players = player_match_check["matched_flag"].sum()
unmatched_players = (~player_match_check["matched_flag"]).sum()

print("\nPlayer-level match result:")
print("Total players:", len(player_match_check))
print("Matched players:", matched_players)
print("Unmatched players:", unmatched_players)

# ----------------------------------------
# 6. quick check
# ----------------------------------------
print("\nSample merged rows:")
print(
    df_with_archetype[
        [
            "player_id",
            "player_name_fixed",
            "year",
            "macro_archetype",
            "shot_style_subtype"
        ]
    ].head(20)
)

# ----------------------------------------
# 7. show unmatched players if needed
# ----------------------------------------
unmatched_players_df = df_with_archetype[
    df_with_archetype["macro_archetype"].isna()
][["player_id", "player_name_fixed"]].drop_duplicates().sort_values("player_name_fixed")

print("\nUnmatched players:")
print(unmatched_players_df)

Rows in block2_small: 1058
Unique players in block2_small: 1058

Row-level match result:
Total rows: 3596
Matched rows: 2240
Unmatched rows: 1356

Player-level match result:
Total players: 899
Matched players: 560
Unmatched players: 339

Sample merged rows:
    player_id player_name_fixed  year                  macro_archetype  \
0      201985          AJ Price  2009     Perimeter Wings & Connectors   
1      201985          AJ Price  2010     Perimeter Wings & Connectors   
2      201985          AJ Price  2011     Perimeter Wings & Connectors   
3      201985          AJ Price  2012     Perimeter Wings & Connectors   
4      201166      Aaron Brooks  2007      High-Usage Primary Creators   
5      201166      Aaron Brooks  2008      High-Usage Primary Creators   
6      201166      Aaron Brooks  2009      High-Usage Primary Creators   
7      201166      Aaron Brooks  2010      High-Usage Primary Creators   
8      203932      Aaron Gordon  2014  Scoring Bigs / Two-Way Forwards   
9 

In [17]:
# keep only matched rows
df_with_archetype_matched = df_with_archetype[
    df_with_archetype["macro_archetype"].notna()
].copy()

print("Original shape:", df_with_archetype.shape)
print("After removing unmatched rows:", df_with_archetype_matched.shape)
print("Unique players after removing unmatched rows:",
      df_with_archetype_matched["player_id"].nunique())

Original shape: (3596, 48)
After removing unmatched rows: (2240, 48)
Unique players after removing unmatched rows: 560


# Step 12: Constructing the Year 5 Salary Target Table

Now we construct the forecast targets for the salary forecasting task. Starting with a pool of players who have played for more than 5 years, we determine each player's **5th season** and retain only that row of data. This will give the forecasting model the expected outcome for year 5.

In [18]:
# -----------------------------
# 1. find the player id column
# -----------------------------
possible_id_cols = ["player_id", "PLAYER_ID", "playerid", "Player_ID"]
id_col = None
for c in possible_id_cols:
    if c in df_5plus.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError("No player id column found in df_5plus.")

# -----------------------------
# 2. sort and define 5th year
# -----------------------------
df_5plus = df_5plus.sort_values(["player_name_fixed", "year"]).copy()
df_5plus["season_num"] = df_5plus.groupby("player_name_fixed").cumcount() + 1

# keep only 5th year
df_y5 = df_5plus[df_5plus["season_num"] == 5].copy()

# -----------------------------
# 3. make salary columns numeric
# -----------------------------
df_y5["salary"] = pd.to_numeric(df_y5["salary"], errors="coerce")
df_y5["salary_cap"] = pd.to_numeric(df_y5["salary_cap"], errors="coerce")

# create salary cap percentage
df_y5["salary_cap_pct"] = df_y5["salary"] / df_y5["salary_cap"]

# -----------------------------
# 4. keep only requested columns
# -----------------------------
df_y5 = df_y5[
    [id_col, "player_name_fixed", "year", "salary", "salary_cap", "salary_cap_pct"]
].copy()

# rename id column to player_id if needed
if id_col != "player_id":
    df_y5 = df_y5.rename(columns={id_col: "player_id"})

# -----------------------------
# 5. check result
# -----------------------------
print(df_y5.columns.tolist())
print(df_y5.head(20))
print("Shape:", df_y5.shape)
print("Unique players:", df_y5["player_name_fixed"].nunique())

['player_id', 'player_name_fixed', 'year', 'salary', 'salary_cap', 'salary_cap_pct']
      player_id player_name_fixed  year      salary   salary_cap  \
5193     201985          AJ Price  2013    947907.0   58679000.0   
4809     201166      Aaron Brooks  2012   5750000.0   58044000.0   
7142     203932      Aaron Gordon  2018  21590909.0  101869000.0   
4426     201189        Aaron Gray  2011   2500000.0   58044000.0   
8704    1628988     Aaron Holiday  2022   1968175.0  123655000.0   
1438        243       Aaron McKie  2003   5000000.0   43840000.0   
9497    1630174     Aaron Nesmith  2024  11000000.0  140588000.0   
9893    1630598     Aaron Wiggins  2025  10102803.0  154647000.0   
1439       1425    Aaron Williams  2003   2925000.0   43840000.0   
1440       1502      Adonal Foyle  2003   4400000.0   43840000.0   
1791       1559    Adrian Griffin  2004    870046.0   43870000.0   
1441       1733     Al Harrington  2003   5692500.0   43840000.0   
4810     201143        Al Horfo

In [19]:
df_y5.to_csv(RESULT_DIR / "df_y5.csv", index=False)
print("Saved file: df_y5.csv")
print(df_y5.shape)

Saved file: df_y5.csv
(899, 6)


# Step 13. Add LSTM confidence information to the final forecasting table

Finally, we merge the confidence information from the LSTM prediction table into the archetype-matched forecasting dataset.

This gives the final forecasting table an extra model-based signal that can later be used for:

- feature design
- interpretation
- uncertainty analysis

At the end of this step, we save the two main output files from this notebook:

- `df_y5.csv` as the Year-5 target table
- `df_with_archetype_matched.csv` as the final cleaned forecasting feature table

In [20]:
# ----------------------------------------
# 1. standardize column names
# ----------------------------------------
df_with_archetype_matched.columns = df_with_archetype_matched.columns.str.strip().str.lower()
final_lstm_predictions.columns = final_lstm_predictions.columns.str.strip().str.lower()

# ----------------------------------------
# 2. find confidence column in final_lstm_predictions
# ----------------------------------------
possible_conf_cols = ["confidence", "max_pred_prob"]
conf_col = None
for c in possible_conf_cols:
    if c in final_lstm_predictions.columns:
        conf_col = c
        break

if conf_col is None:
    raise ValueError(
        f"No confidence column found in final_lstm_predictions. Available columns: {final_lstm_predictions.columns.tolist()}"
    )

# ----------------------------------------
# 3. keep only player_id + confidence
# ----------------------------------------
lstm_conf = final_lstm_predictions[["player_id", conf_col]].copy()
lstm_conf = lstm_conf.drop_duplicates(subset=["player_id"]).copy()
lstm_conf = lstm_conf.rename(columns={conf_col: "lstm_confidence"})

print("Rows in lstm_conf:", len(lstm_conf))
print("Unique player_id in lstm_conf:", lstm_conf["player_id"].nunique())

# ----------------------------------------
# 4. merge into df_with_archetype_matched
# ----------------------------------------
df_with_archetype_matched = df_with_archetype_matched.merge(
    lstm_conf,
    on="player_id",
    how="left"
)

# ----------------------------------------
# 5. quick checks
# ----------------------------------------
print("Rows after merge:", len(df_with_archetype_matched))
print("Matched confidence rows:", df_with_archetype_matched["lstm_confidence"].notna().sum())
print("Unmatched confidence rows:", df_with_archetype_matched["lstm_confidence"].isna().sum())

print(
    df_with_archetype_matched[
        ["player_id", "player_name_fixed", "year", "lstm_confidence"]
    ].head(20)
)

Rows in lstm_conf: 1121
Unique player_id in lstm_conf: 1121
Rows after merge: 2240
Matched confidence rows: 2240
Unmatched confidence rows: 0
    player_id player_name_fixed  year lstm_confidence
0      201985          AJ Price  2009            High
1      201985          AJ Price  2010            High
2      201985          AJ Price  2011            High
3      201985          AJ Price  2012            High
4      201166      Aaron Brooks  2007          Medium
5      201166      Aaron Brooks  2008          Medium
6      201166      Aaron Brooks  2009          Medium
7      201166      Aaron Brooks  2010          Medium
8      203932      Aaron Gordon  2014            High
9      203932      Aaron Gordon  2015            High
10     203932      Aaron Gordon  2016            High
11     203932      Aaron Gordon  2017            High
12     201189        Aaron Gray  2007            High
13     201189        Aaron Gray  2008            High
14     201189        Aaron Gray  2009           

In [21]:
df_with_archetype_matched.to_csv(RESULT_DIR / "df_with_archetype_matched.csv", index=False)
print("\nSaved: df_with_archetype_matched.csv")
print(df_with_archetype_matched.shape)


Saved: df_with_archetype_matched.csv
(2240, 49)
